# pdserve — Real-GPU Validation (Colab T4)

Validates pdserve's roofline-simulator claims on real hardware: **Scenario A** (colocated burst → TPOT inflation), **Scenario B** (prefix caching win), **Scenario D** (quantization tradeoff).

> T4 = 1 GPU, 16GB → we run **Qwen2.5-7B AWQ (4-bit)** + **Qwen2.5-3B FP16**. True PD disaggregation (Scenario C) needs 2 GPUs — see `benchmark/GPU_PLAN.md` for the A10 run.

**Runtime required: GPU T4** (Runtime → Change runtime type).

In [ ]:
# 1) Environment check
!nvidia-smi
import torch, subprocess, json, time, threading, random, statistics, os
print("torch:", torch.__version__)

In [ ]:
# 2) Install vLLM (pinned; record the exact version)
!pip install -q "vllm>=0.8,<0.9" openai
import vllm
print("vllm:", vllm.__version__)

In [ ]:
# 3) Launch vLLM server (colocated baseline, AWQ 7B)
from openai import OpenAI
import subprocess, requests

MODEL_AWQ = "Qwen/Qwen2.5-7B-Instruct-AWQ"
SERVER = "http://localhost:8000"

server_log = open("/tmp/vllm.log", "w")
proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL_AWQ,
     "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.92"],
    stdout=server_log, stderr=subprocess.STDOUT)

for _ in range(120):  # up to 10 min (model download on first run)
    time.sleep(5)
    try:
        if requests.get(SERVER + "/health", timeout=2).status_code == 200:
            print("server up"); break
    except Exception:
        pass
else:
    print("SERVER FAILED — tail of log:"); print(open("/tmp/vllm.log").read()[-2000:])

client = OpenAI(base_url=SERVER + "/v1", api_key="empty")

In [ ]:
# 4) Load generator: Poisson arrivals + burst, streaming TTFT/TPOT capture
def one_request(client, model, prompt, out_tokens=128):
    t0 = time.time(); ttft = None; itls = []
    try:
        stream = client.completions.create(model=model, prompt=prompt,
                                           max_tokens=out_tokens, temperature=0.0, stream=True)
        prev = t0
        for chunk in stream:
            now = time.time()
            if ttft is None: ttft = now - t0
            else: itls.append(now - prev)
            prev = now
        return {"ttft": ttft, "tpot": (sum(itls)/len(itls)*1000) if itls else 0,
                "e2e": time.time() - t0, "ok": True}
    except Exception as e:
        return {"ttft": ttft or 0, "tpot": 0, "e2e": time.time() - t0, "ok": False, "err": str(e)[:80]}

def run_load(client, model, rps=2.0, dur=30, burst_at=10, burst_factor=6, burst_len=3,
             prompt="You are a helpful assistant. " * 60):
    arrivals = []; t = 0.0
    while t < dur:
        dt = random.expovariate(rps * (burst_factor if burst_at <= t < burst_at + burst_len else 1.0))
        t += dt; arrivals.append(min(t, dur))
    results = []; lock = threading.Lock()
    def worker(at):
        time.sleep(at)
        r = one_request(client, model, prompt)
        with lock: results.append({"at": at, **r})
    threads = [threading.Thread(target=worker, args=(a,)) for a in arrivals]
    for th in threads: th.start()
    for th in threads: th.join()
    ok = [r for r in results if r["ok"]]
    return {
        "sent": len(results), "ok": len(ok),
        "ttft_p99": round(sorted(r["ttft"] for r in ok)[int(len(ok)*0.99)-1], 3) if ok else None,
        "tpot_p50_ms": round(statistics.median(r["tpot"] for r in ok), 2) if ok else None,
        "tpot_p99_ms": round(sorted(r["tpot"] for r in ok)[int(len(ok)*0.99)-1], 2) if ok else None,
    }

print("generator ready")

In [ ]:
# 5) SCENARIO A — colocated burst (the 133x claim, on real hardware)
# idle baseline first, then burst run
random.seed(42)
base = run_load(client, MODEL_AWQ, rps=1.0, dur=20, burst_at=999)      # no burst
random.seed(42)
burst = run_load(client, MODEL_AWQ, rps=1.0, dur=30, burst_at=10, burst_factor=6)
print("IDLE :", base)
print("BURST:", burst)
print("TPOT p99 inflation: %.1fx" % (burst["tpot_p99_ms"] / max(base["tpot_p99_ms"], 0.01)))
scenario_a = {"idle": base, "burst": burst}

In [ ]:
# 6) SCENARIO B — prefix caching ON vs OFF (shared 2K-token system prompt)
SYS = "You are a financial analyst. " * 300   # long shared prefix
Q  = "Summarize the risk factors."
def shared_prefix_run(enable):
    # restart server with the flag
    global proc
    proc.terminate(); proc.wait()
    server_log = open("/tmp/vllm.log", "a")
    args = ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL_AWQ,
            "--max-model-len", "4096", "--gpu-memory-utilization", "0.92"]
    if enable: args.append("--enable-prefix-caching")
    proc = subprocess.Popen(args, stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(90):
        time.sleep(5)
        try:
            if requests.get(SERVER + "/health", timeout=2).status_code == 200: break
        except Exception: pass
    random.seed(7)
    return run_load(client, MODEL_AWQ, rps=1.0, dur=20, burst_at=999, prompt=SYS + Q)

off = shared_prefix_run(False)
on  = shared_prefix_run(True)
print("prefix-cache OFF:", off)
print("prefix-cache ON :", on)
scenario_b = {"off": off, "on": on}

In [ ]:
# 7) SCENARIO D (lite) — FP16 3B vs AWQ 7B: quality-tier vs throughput-tier
MODEL_3B = "Qwen/Qwen2.5-3B-Instruct"
proc.terminate(); proc.wait()
proc = subprocess.Popen(["python", "-m", "vllm.entrypoints.openai.api_server",
                         "--model", MODEL_3B, "--max-model-len", "4096",
                         "--gpu-memory-utilization", "0.92"],
                        stdout=open("/tmp/vllm.log","a"), stderr=subprocess.STDOUT)
for _ in range(90):
    time.sleep(5)
    try:
        if requests.get(SERVER + "/health", timeout=2).status_code == 200: break
    except Exception: pass
random.seed(42)
fp16_3b = run_load(client, MODEL_3B, rps=1.0, dur=20, burst_at=999)
print("3B FP16:", fp16_3b)
scenario_d = {"fp16_3b": fp16_3b, "awq_7b": scenario_a["idle"]}

In [ ]:
# 8) Save results — download this file and drop it into pdserve/benchmark/results/
import json
out = {
  "hardware": "Colab T4 (16GB)", "engine": "vLLM " + vllm.__version__,
  "date": time.strftime("%Y-%m-%d"),
  "scenarios": {"A_colocated_burst": scenario_a, "B_prefix_cache": scenario_b, "D_quant_tiers": scenario_d},
}
json.dump(out, open("gpu_t4_results.json", "w"), indent=2)
print(open("gpu_t4_results.json").read())

## Publish back

1. Download `gpu_t4_results.json` from the Colab file panel.
2. Drop it into `pdserve/benchmark/results/`.
3. Update `benchmark/README.md`: add a **"Real-GPU (T4) validation"** section next to the simulator tables, comparing:
   - Scenario A: measured TPOT p99 inflation vs the simulator's 133× claim
   - Scenario B: TTFT delta with prefix caching ON/OFF
   - Scenario D: FP16-3B vs AWQ-7B throughput/latency
4. PR or push to `main`.

> Honesty rules: report the numbers you got, even if the inflation is smaller than the sim predicts — the sim-vs-real delta is itself the story (see `GPU_PLAN.md` §7).